# IgniteCyber 5‑Day Cyber AI Analyst Bootcamp — LAB2.3

**Offline-only** • **Sigma-first** • **Elastic/Kibana + TheHive 5 + Cortex + MISP**  
**Datasets:** `/opt/bootcamp/datasets` • **Notebooks:** `/opt/bootcamp/notebooks`

## CJ Intel Spine (Single evolving MISP Event + one Event Report)
Throughout the week, we maintain **one** MISP event for the threat actor **Cinder Jackal (CJ)** and append a **single MISP Event Report** day-by-day.

**Workflow (Notebook-guided / manual in TheHive UI):**
1. Hunt/confirm in **Kibana** (`zeek-*`, `wazuh-alerts-*`, `elastic-agent-*`)
2. Add observables + evidence to **TheHive case**
3. Run **Cortex analyzers** from TheHive (sightings / file triage)
4. Promote validated intel into **MISP objects**
5. Append the narrative block generated by this notebook into the **CJ MISP Event Report**

## Required dataset bundles for this lab
- (see course dataset catalog for this lab bundle)


# Lab 2.3 — LOLBAS Proxy Execution: mshta.exe (T1218.005)


## Scenario storyline (persistent across the week)

**Company:** ApexFin Solutions (AF) — a fast-growing fintech startup that provides a consumer budgeting app and small-business payment rails.

**Business critical assets:**
- AF-PAY API (customer payments)
- AF-PORTAL (employee/admin portal)
- AF-ANALYTICS (data warehouse exports)

**Threat actor:** CINDER JACKAL (fictional) — financially motivated intrusion set that blends commodity tradecraft with “living-off-the-land” execution and quick monetization.

**CINDER JACKAL objectives:**
1) Obtain employee credentials via phishing and browser session theft  
2) Establish persistence and stealthy C2  
3) Identify and exfiltrate customer PII and payment reconciliation data  
4) Extort FinanceFront with data-theft + disruption pressure

We use **MITRE ATT&CK** to describe behavior, **Sigma-first detections** for portability, and **Elastic/Kibana + Wazuh/Zeek** telemetry for investigations.



### Lab VM / Data assumptions (offline-only)

- Datasets (ZIP) are available locally in: `/opt/bootcamp/datasets`
- Elastic/Kibana index patterns:
  - `zeek-*`
  - `wazuh-alerts-*`
  - `elastic-agent-*`
- Elastic user/pass are preconfigured in the VM (read from environment variables by default).
- Local LLM: **Ollama** on `http://localhost:11434` (no internet required).


## Objectives
- Complete the tasks for **Lab 2.3** and capture evidence.
- Map observed behavior to MITRE ATT&CK.
- Produce a Sigma-first detection outcome.

## MITRE ATT&CK mapping
- **TA0005 / T1218.005** — Mshta

## Datasets (offline ZIPs)
- Mshta VBScript Execute PowerShell *(hint: `mshta_vbscript_execute_psh`)* — mapped to **T1218.005**
- Mshta Javascript GetObject Sct *(hint: `mshta_javascript_getobject_sct`)* — mapped to **T1218.005**
- Mshta HTML Application (HTA) Execution *(hint: `mshta_html_application_execution`)* — mapped to **T1218.005**

## Tasks (do these in order)
1. Find mshta execution events and identify their parent processes.
1. Determine the payload source (local .hta vs remote scriptlet / URL).
1. Correlate endpoint execution with Zeek DNS/HTTP around the same time window.
1. Write a Sigma rule and validate it against your evidence.


> Attribution / inspiration: see `docs/references.md` (Security Break Jupyter Collection + Security Datasets).


In [ ]:
import os
from bootcamp_lib import *
print('ES_HOST:', ES_HOST)
print('Ollama models:', ollama_models()[:5])
print('Dataset ZIP count:', len(list_dataset_zips()))


## Step 1 — Elastic health checks


In [ ]:
info = es_get('/')
print(json.dumps({k: info.get(k) for k in ['name','cluster_name','cluster_uuid','version','tagline']}, indent=2))
health = es_get('/_cluster/health')
print(json.dumps({k: health.get(k) for k in ['status','number_of_nodes','active_primary_shards','active_shards','unassigned_shards']}, indent=2))


## Step 2 — Confirm index patterns exist (zeek, wazuh, elastic-agent)


In [ ]:
aliases = es_get('/_aliases')
idx = sorted(list(aliases.keys()))
patterns = ['zeek-','wazuh-alerts-','elastic-agent-']
found = {p:[i for i in idx if i.startswith(p)] for p in patterns}
for p, items in found.items():
    print(p, '=>', len(items), 'indices')
    for i in items[:10]:
        print('  -', i)


## Step 3 — Locate the dataset ZIP for this lab (offline)
If a hint does not match any ZIP filename, list all ZIPs and pick the closest one.


In [ ]:
hints = ['mshta_vbscript_execute_psh', 'mshta_javascript_getobject_sct', 'mshta_html_application_execution']
print('All ZIPs (first 30):')
print('\n'.join(list_dataset_zips()[:30]))

zips=[]
for h in hints:
    z = find_dataset_zip(h)
    if z:
        zips.append(z)
print('\nCandidate ZIPs:')
for z in zips:
    print(' -', z)
if not zips:
    print('No matching ZIP found. Pick a ZIP manually and set zip_path = <path>.')


## Step 4 — Inspect the first JSONL inside the ZIP (preview events)


In [ ]:
hints = ['mshta_vbscript_execute_psh', 'mshta_javascript_getobject_sct', 'mshta_html_application_execution']
zip_path=None
for h in hints:
    z = find_dataset_zip(h)
    if z:
        zip_path=z
        break
if zip_path:
    member, df_ds = read_first_jsonl_from_zip(zip_path, max_lines=5000)
    print('ZIP:', zip_path)
    print('Member:', member)
    display(df_ds.head(10))
    print('Columns (first 30):', list(df_ds.columns)[:30])
else:
    print('Dataset ZIP not found. Provide zip_path manually.')


## Step 5 — Sigma-first detection

### Sigma-first workflow (how we use Sigma in this bootcamp)

1) Start with a **Sigma rule** (portable detection logic).  
2) Translate to **KQL** (Kibana) and/or Elastic Query DSL for execution.  
3) Validate against local datasets (offline) and then tune for noise.

#### Sigma rule (authoritative)

```yaml
title: FinanceFront Mshta Proxy Execution
id: 1a4b5a6b-ff_bootcamp_2_3
status: experimental
description: Detects mshta proxy execution patterns used for initial payload execution.
author: IgniteCyber Academy
date: 2026/02/02
logsource:
  category: process_creation
  product: windows
detection:
  selection_img:
    Image|endswith: '\mshta.exe'
  selection_cmd:
    CommandLine|contains:
      - 'vbscript:Execute('
      - 'javascript:'
      - '.hta'
      - 'GetObject('
  condition: selection_img and selection_cmd
falsepositives:
  - Rare legitimate HTA usage
level: high
tags:
  - attack.defense_evasion
  - attack.t1218.005
```

#### Equivalent Kibana KQL (recommended for students)

```text
process.name:mshta.exe and process.command_line:(*vbscript:Execute(* or *javascript:* or *.hta* or *GetObject(* )
```

#### Quick query_string pivot (used by this notebook)

```text
(process.name:mshta.exe OR winlog.event_data.Image:*mshta.exe) AND ("vbscript:Execute(" OR "javascript:" OR ".hta" OR "GetObject(")
```


In [ ]:
qs = '(process.name:mshta.exe OR winlog.event_data.Image:*mshta.exe) AND ("vbscript:Execute(" OR "javascript:" OR ".hta" OR "GetObject(")'
res = es_qs('elastic-agent-*,wazuh-alerts-*', qs, size=200)
df = hits_to_df(res)
print('Events returned:', len(df))
display(df.head(20))


## Step 6 — Starter investigation queries (copy/paste into Kibana)
- `process.name:mshta.exe`
- `process.command_line:(*vbscript:Execute(* or *javascript:* or *.hta* or *GetObject(* )`


## Step 7 — Build an evidence packet (for reporting + AI assistance)
Adjust `sample_qs` to match what you found in Steps 3–6.


In [ ]:
sample_qs = 'process.name:(powershell.exe OR mshta.exe OR bitsadmin.exe) OR event.dataset:zeek.conn'
res = es_qs('elastic-agent-*,wazuh-alerts-*,zeek-*', sample_qs, size=500)
df = hits_to_df(res)
out_path = save_evidence_csv(df, 'Lab_2.3')
print('Saved evidence CSV:', out_path, 'rows:', len(df))
display(df.head(15))


## Step 8 — AI assistance (local Ollama) + human verification
**AI task for this lab:** Ask Ollama to generate 3 tuning variants (strict, medium, broad) and explain tradeoffs.

Rule: **No evidence → no claims.** Validate output with additional queries.


In [ ]:
models = ollama_models()
model = models[0] if models else "llama3"
evidence_preview = df.head(30).to_csv(index=False) if 'df' in globals() and isinstance(df, pd.DataFrame) else ""
prompt = f'''You are a SOC threat hunter. Work offline. Use only the evidence provided.

OUTPUT FORMAT (strict):
- Summary (5 bullets)
- ATT&CK techniques (ID + 1-line justification referencing evidence columns)
- Validation queries (3 KQL queries)
- Containment actions (2)
- Uncertainty (what is missing / what could be wrong)

EVIDENCE (CSV excerpt):
{evidence_preview}
'''
if evidence_preview.strip():
    response = ollama_generate(prompt, model=model, temperature=0.2)
    print(response)
else:
    print("No evidence dataframe found yet. Run Step 7 first.")


## Deliverables / Checkpoint
- Evidence CSV path (saved in `/tmp/`)
- ATT&CK technique IDs you verified
- Sigma rule or tuning notes (if applicable)
- 1-paragraph narrative (what happened + why you believe it)


## CJ Event Report (copy/paste into MISP → Event → Event Reports)

**Report name (single for the week):** `Cinder Jackal — Weekly Narrative (Bootcamp)`  
In MISP: open the **CJ Event** → **Event Reports** → open the weekly report → paste the block below at the end.

> Tip: Keep headings consistent so the report reads like a timeline.

### Paste this block


In [ ]:
from datetime import datetime
import os, textwrap

LAB_CODE = "LAB2.3"
DAY_LABEL = os.getenv("BOOTCAMP_DAY", "") or ""
TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ")

# Fill these as you work the lab (concise + evidence-driven)
findings = {
    "summary": "",
    "what_happened": [],
    "key_iocs": [],
    "attack_mapping": [],
    "elastic_evidence": [],
    "thehive_actions": [],
    "misp_updates": [],
    "next_steps": [],
}

def _lines(items, fallback="(fill in)"):
    return "\n".join(["- " + x for x in (items if items else [fallback])])

def render_report_block():
    md = f"""\
## {DAY_LABEL + " — " if DAY_LABEL else ""}{LAB_CODE} — CJ Narrative Update
**Timestamp (UTC):** {TS}

**Summary:** {findings["summary"] or "(fill in)"}

**What happened**
{_lines(findings["what_happened"])}

**Key IOCs**
{_lines(findings["key_iocs"])}

**MITRE ATT&CK mapping**
{_lines(findings["attack_mapping"])}

**Elastic evidence (queries / fields / indices)**
{_lines(findings["elastic_evidence"])}

**TheHive actions**
{_lines(findings["thehive_actions"])}

**MISP updates (objects + tags)**
{_lines(findings["misp_updates"])}

**Next steps**
{_lines(findings["next_steps"])}

---"""
    return textwrap.dedent(md)

report_block = render_report_block()
print(report_block)

out_dir = "/opt/bootcamp/reports/cj"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f"{LAB_CODE.replace('.','_')}_cj_event_report_update.md")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_block)

print(f"Saved: {out_path}")


## Daily Executive Summary Checkpoint

Use this checkpoint at the end of each lab day to build the rolling executive summary for the Cinder Jackal investigation. The AI assistant can help turn validated evidence into executive-ready language, but it must not introduce new facts.

**Required workflow:**
1. Confirm the lab evidence packet or report block is filled in.
2. Ask the local AI assistant to draft a concise executive update from that evidence only.
3. Validate every claim against notebook output, Elastic/OpenSearch results, TheHive tasks, MISP updates, and source artifacts.
4. Save the final daily checkpoint under `/opt/bootcamp/reports/cj/` and append approved content to the weekly CJ Event Report.


In [ ]:
# Daily Executive Summary Checkpoint - AI-assisted, evidence-first
from datetime import datetime
import os, textwrap, json

_checkpoint_lab = globals().get("LAB_CODE", "LAB-UNKNOWN")
_checkpoint_day = globals().get("DAY_LABEL", "") or os.getenv("BOOTCAMP_DAY", "") or "Day checkpoint"
_checkpoint_ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ")

# Prefer the validated CJ Event Report block when the notebook has one.
_checkpoint_evidence = ""
if "report_block" in globals() and str(report_block).strip():
    _checkpoint_evidence = str(report_block)
elif "findings" in globals():
    _checkpoint_evidence = json.dumps(findings, indent=2)
elif "report" in globals():
    _checkpoint_evidence = json.dumps(report, indent=2)
else:
    _checkpoint_evidence = "No structured report block found yet. Fill in the lab findings before finalizing this checkpoint."

_daily_exec_prompt = f"""
You are assisting a SOC analyst with an executive update for the IgniteCyber bootcamp.
Use only the validated evidence below. Do not invent facts, impacts, victims, or containment actions.
If evidence is missing, say what must be validated before leadership reporting.

Return this exact structure:
1. Executive summary: 3-5 bullets, business-focused, no jargon unless necessary.
2. What changed today: short paragraph.
3. Current risk: Low/Medium/High with evidence-based justification.
4. Validated evidence: 3-6 bullets with field names, artifacts, or source systems.
5. Decisions or actions needed: 2-4 bullets.
6. Open questions: 2-4 bullets.
7. Analyst validation checklist: claims that must be checked before sending.

Lab: {_checkpoint_lab}
Day: {_checkpoint_day}
Timestamp UTC: {_checkpoint_ts}

VALIDATED EVIDENCE PACKET:
{_checkpoint_evidence}
""".strip()

_daily_template = f"""# {_checkpoint_day} - Executive Summary Checkpoint

**Lab:** {_checkpoint_lab}  
**Timestamp (UTC):** {_checkpoint_ts}  
**Status:** Draft - requires analyst validation

## Executive Summary
- (AI-assisted draft or analyst-written summary goes here.)

## What Changed Today
- (Summarize the investigation progress from this lab.)

## Current Risk
- (Low/Medium/High and why.)

## Validated Evidence
- (Cite source systems, fields, observables, or artifacts.)

## Decisions or Actions Needed
- (List leadership or SOC actions.)

## Open Questions
- (List unknowns and validation gaps.)

## Analyst Validation Checklist
- [ ] Claims match source evidence.
- [ ] No AI-only claims remain.
- [ ] IOCs are validated before MISP promotion.
- [ ] TheHive tasks reflect current status.
- [ ] Executive wording is accurate and appropriately scoped.
"""

print(_daily_template)
print("\n--- AI prompt prepared for local assistant ---\n")

_models = ollama_models() if "ollama_models" in globals() else []
if _models and "ollama_stream" in globals():
    print(f"Using Ollama model: {_models[0]}\n")
    ollama_stream(_daily_exec_prompt, model=_models[0], temperature=0.2)
elif _models and "ollama_generate" in globals():
    print(f"Using Ollama model: {_models[0]}\n")
    print(ollama_generate(_daily_exec_prompt, model=_models[0], temperature=0.2))
else:
    print("No local Ollama helper/model is available in this notebook session.")
    print("Use the template above, or paste _daily_exec_prompt into the approved local AI workflow.")

_out_dir = "/opt/bootcamp/reports/cj"
os.makedirs(_out_dir, exist_ok=True)
_safe_lab = _checkpoint_lab.replace(".", "_").replace(" ", "_").replace("-", "_")
_out_path = os.path.join(_out_dir, f"{_safe_lab}_daily_executive_summary_checkpoint.md")
with open(_out_path, "w", encoding="utf-8") as f:
    f.write(_daily_template)

print(f"\nSaved daily executive summary checkpoint template: {_out_path}")
